In [ ]:
from pathlib import Path

folder = Path("/scratch/yag1/ego4d_data/v2/clips")

subdirs = [p.name for p in folder.iterdir() if p.is_dir()]
print(subdirs)

In [ ]:
print(len(subdirs))

In [ ]:
import pandas as pd

In [ ]:
text_embeddings_df = pd.read_csv("/scratch/yag1/ego4d_data/v2/queries_with_talk_and_paths.csv")

In [ ]:
text_embeddings_df.head()

In [ ]:
text_embeddings_df

In [ ]:
embedding_dir = "video_embeddings"

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
def norm(embeddings):
  return F.normalize(embeddings, p=2, dim=1)# along each row

In [ ]:
video_to_embedding = {}

In [ ]:
for subdir in subdirs:
    video_embedding_path = Path(folder / subdir / embedding_dir)
    embedding_files = [p for p in video_embedding_path.iterdir() if p.is_file()]
    video_to_embedding[subdir] = [[], embedding_files]
    print(f"Processing {subdir} with {len(embedding_files)} embedding files")
    for embedding_file in embedding_files:
        video_to_embedding[subdir][0].append(torch.load(video_embedding_path / embedding_file))
    video_to_embedding[subdir][0] = norm(torch.stack(video_to_embedding[subdir][0], dim=0))
    print(video_to_embedding[subdir][0].shape)    

In [ ]:
embeddings = video_to_embedding[subdir][0]   # shape: [num_embeddings, dim]
norms = torch.linalg.norm(embeddings, dim=1)

print(norms[:10])
print("min:", norms.min().item())
print("max:", norms.max().item())
print("all close to 1:", torch.allclose(norms, torch.ones_like(norms), atol=1e-5))

In [ ]:
import faiss
import numpy as np

In [ ]:
for k,v in video_to_embedding.items():
    base_index = faiss.IndexFlatIP(v[0].shape[1])  # dimension of embeddings
    faiss_index = faiss.IndexIDMap(base_index)
    
    id_array = np.arange(v[0].shape[0])  # unique IDs for each embedding
    faiss_index.add_with_ids(v[0].numpy(), id_array)
    
    video_to_embedding[k].append(faiss_index)

In [ ]:
import ast

In [ ]:
text_embeddings_df["video_prism_distances"] = None
text_embeddings_df["video_prism_indices"] = None
text_embeddings_df["video_prism_min_distance"] = None
text_embeddings_df["video_prism_max_distance"] = None

for clip_id, v in video_to_embedding.items():
    matching_rows = text_embeddings_df[
        text_embeddings_df["clip_id"] == clip_id
    ]

    for idx in matching_rows.index:
        path = text_embeddings_df.at[idx, "video_prism_text_embedding_path"]

        query = norm(torch.load(path).unsqueeze(0))
        
        D, I = v[2].search(query.numpy(), k=(v[2].ntotal))
        
        print(len(I[0]), len(D[0]), v[2].ntotal, clip_id, idx)

        text_embeddings_df.at[idx, "video_prism_distances"] = D[0].tolist()
        text_embeddings_df.at[idx, "video_prism_indices"] = I[0].tolist()
        text_embeddings_df.at[idx, "video_prism_max_distance"] = D[0].max().item()
        text_embeddings_df.at[idx, "video_prism_min_distance"] = D[0].min().item()
        

In [ ]:
text_embeddings_df.head()

In [ ]:
text_embeddings_df["video_prism_indices_unique"] = text_embeddings_df["video_prism_indices"].apply(
    lambda x: len(x) == len(set(x))
)

In [ ]:
text_embeddings_df["video_prism_indices_unique"].value_counts()

In [ ]:
text_embeddings_df.to_csv("/scratch/yag1/ego4d_data/v2/queries_with_talk_and_paths_and_video_retrieval.csv", index=False)